In [1]:
!pip install ollama

In [2]:
!sudo apt update
!usdo apt install -y pciutils
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/bin/bash: line 1: usdo: command not found
Reading pac

In [11]:
from asyncio.unix_events import subprocess
import time
import ollama
import asyncio
import nest_asyncio
import os
import requests
from transformers import pipeline
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import re
from typing import Optional, List, Dict, Any

import torch
import multiprocessing
from sklearn.metrics import accuracy_score, f1_score

nest_asyncio.apply()


class OllamaModelTester:

    def __init__(self, host, port, models):
        self.host = host
        self.port = port
        self.models = models if models else []
        self.process = None
        self.results = []
        self.dft_sleep_sec = 10
        self.nli_model = None
        self.initialization()

    def initialization(self):
        os.environ['OLLAMA_HOST'] = f'{self.host}:{self.port}'

        try:
            nltk.data.find('tokenizers/punkt_tab/english')
        except LookupError:
            nltk.download('punkt_tab')
            nltk.download('punkt')
            nltk.download('averaged_perceptron_tagger_eng')

        self.nli_model = pipeline('text-classification',
            model='cross-encoder/nli-deberta-v3-base',
            device=0
        )

        print('Class OllamaModelTester is initialized')

    def start_server(self):
        self.process = subprocess.Popen(
            ['ollama', 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ
        )
        time.sleep(self.dft_sleep_sec)
        result = subprocess.run(
            ['curl', '-s', f'http://{self.host}:{self.port}/api/tags'],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print('Ollama server is started')
        else:
            print('Ollama server is NOT started')

    def stop_server(self):
        self.process.terminate()
        self.process.wait()
        print('Ollama server is terminated')

    def pull_model(self, model_name):
        result = subprocess.run(
            ['ollama', 'pull', model_name],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print(f'"{model_name}" model is pulled successfully!')
        else:
            print(f'"{model_name}" model pull threw an error')

    def pull_models(self, models):
        models = models if models else self.models
        for model_name in models:
            result = subprocess.run(
                ['ollama', 'pull', model_name],
                capture_output=True,
                text=True
            )
            if (result.returncode == 0):
                print(f'"{model_name}" model is pulled successfully!')
            else:
                print(f'"{model_name}" model pull threw an error')

    def compare_models(self, prompt_text, models, **options):
        var_models = models if models else self.models
        var_options = {
            'temperature': options.get('temperature', 0.1),
            'num_ctx': options.get('num_ctx', 512)
        }
        for model_name in var_models:
            try:
                print(f'Testing model "{model_name}"')

                first_token_received = False
                tokens_received = 0
                token_times = []
                response_text = ''

                ttft_start = time.time()
                start=time.time()

                response_generator = ollama.generate(
                    model=model_name,
                    prompt=prompt_text,
                    options=var_options,

                    stream=True
                )

                for chunk in response_generator:
                    current_time = time.time()
                    tokens_received += 1

                    if (not first_token_received):
                        ttft = current_time - ttft_start
                        first_token_received = True
                        token_times.append(ttft)
                    else:
                        token_times.append(current_time - start)

                    response_text += chunk['response']

                elapsed=time.time() - start

                tokens_per_second = tokens_received / elapsed if elapsed > 0 else 0
                sorted_times = sorted(token_times)
                p50 = sorted_times[len(sorted_times) // 2] if len(sorted_times) > 0 else 0
                p95 = sorted_times[int(len(sorted_times) * 0.95)] if len(sorted_times) > 0 else 0

                model_comparison: Dict[str, Any] = {
                    'model': model_name,
                    'response': response_text,
                    'tokens': len(response_text.split()),
                    'elapsed_time': round(elapsed, 2),
                    'ttft': round(ttft, 3),
                    'tokens_per_second': round(tokens_per_second, 3),
                    'p50_latency': round(p50, 3),
                    'p95_latency': round(p95, 3)
                }

                # Extract hardware and model information
                try:
                    model_info = self.__get_model_info(model_name)
                except:
                    model_info = {
                        'hardware': 'unknown',
                        'quantization': 'unknown',
                        'param_size': 'unknown',
                        'files_size': 'unknown',
                        'context_length': 'unknown'
                    }
                model_comparison.update(model_info)

                calculate_scores = self.__calculate_scores(prompt_text, response_text)
                model_comparison.update(calculate_scores)

                self.results.append(model_comparison)
                print(f'Successfully completed testing model "{model_name}"')
            except Exception as e:
                self.results.append({
                    'model': model_name,
                    'response': f'"{model_name}" error: {str(e)}',
                    'tokens': 0,
                    'elapsed_time': 0,
                    'ttft': 0,
                    'tokens_per_second': 0,
                    'p50_latency': 0,
                    'p95_latency': 0,
                    'hardware': 'unknown',
                    'quantization': 'unknown',
                    'param_size': 'unknown',
                    'files_size': 'unknown',
                    'context_length': 'unknown'
                })
                print(f'Testing model "{model_name}" threw an error')
        return self.results

    def validate_evaluator(self, prompt_text: str = None, generated_text: str = None, human_label: str = None) -> Dict[str, Any]:
        """
        Meeting 2 (Extended)
        Public method: Validate human generated text and label

        Args:
            prompt_text (str): Original text
            generated_text (str): Human generated text
            human_label (str): Text defining that generated_text is faithful or hallucinated
        Returns:
            Dict[str, Any]
        """
        result: Dict[str, Any] = {}
        try:
            calculate_scores = self.__calculate_scores(prompt_text, generated_text)
            predicted = 'hallucinated' if (calculate_scores['is_hallucinated'] == 1.0) else 'faithful'

            human_labels = [human_label]
            predictions = [predicted]

            overall_accuracy = accuracy_score(human_labels, predictions)
            overall_f1_score = f1_score(human_labels, predictions, pos_label='hallucinated', average='weighted')

            result.update({
                'human_label': human_label,
                'predicted': predicted,
                'overall_accuracy': overall_accuracy,
                'overall_f1_score': overall_f1_score
            })

            # Extract hardware and model information
            try:
                model_info = self.__get_model_info()
            except:
                model_info = {
                    'hardware': 'unknown',
                    'quantization': 'unknown',
                    'param_size': 'unknown',
                    'files_size': 'unknown',
                    'context_length': 'unknown'
                }
            result.update(model_info)
            result.update(calculate_scores)
        except Exception as e:
            print(f'Exception thrown: {str(e)}')
        finally:
            return result

    def print_results(self, i_results: List[Dict[str, Any]] = []):
        var_results = i_results if i_results else self.results
        for row in var_results:
            print('')
            [print(f'{k}: {v}') for k, v in row.items()]
            print('')

    def __clean_text(self, text):
        """
        Meeting 2
        Private method: Preprocess text for tokenization

        Args:
            text(str): Text to preprocess

        Returns:
            str: Preprocessed text
        """
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = ' '.join(text.split())
        return text

    def __calculate_scores(self, reference_text, generated_text):
        """
        Meeting 2
        Private method: Calculate multiple BLUE score variants

        Args:
            reference_text(str): Text passed from main
            generated_text(str): Text generated from AI agent

        Returns:
            Dict[str, float]

        """
        dict_scores = {
            'total': 0.0,
            'factual': 0.0,
            'contradictions': 0.0,
            'neutral': 0.0,
            'confidence': 0.0,
            'is_hallucinated': 0.0,
            'hallucination_severity': 0.0,
            'faithfulness_score': 0.0,
            'bleu_1': 0.0,
            'bleu_2': 0.0,
            'bleu_3': 0.0,
            'bleu_4': 0.0,
            'bleu_avg': 0.0,
            'bleu_smoothing_method0': 0.0,
            'bleu_smoothing_method1': 0.0,
            'bleu_smoothing_method2': 0.0,
            'bleu_smoothing_method3': 0.0,
            'bleu_smoothing_method4': 0.0,
            'bleu_smoothing_method5': 0.0,
            'bleu_smoothing_method6': 0.0,
            'bleu_smoothing_method7': 0.0
        }
        try:
            if (not generated_text or len(generated_text.strip()) == 0):
                return dict_scores

            cleaned_ref = self.__clean_text(reference_text)
            cleaned_gen = self.__clean_text(generated_text)

            reference_tokens = word_tokenize(cleaned_ref)
            generated_tokens = word_tokenize(cleaned_gen)

            if (not reference_tokens or not generated_tokens):
                return dict_scores

            sentences = nltk.sent_tokenize(generated_text)
            if not sentences:
                return dict_scores

            result_scores = []
            for sentence in sentences:
                pred = self.nli_model(f'{reference_text} </s> {sentence}')
                label = (pred[0]['label']).upper()
                score = pred[0]['score']

                result_scores.append({
                    'sentence': sentence,
                    'label': label,
                    'confidence': score
                })

            dict_scores['total'] = len(result_scores)
            dict_scores['factual'] = sum(1 for r in result_scores if r['label'] == 'ENTAILMENT') / len(result_scores)
            dict_scores['contradictions'] = sum(1 for r in result_scores if r['label'] == 'CONTRADICTION') / len(result_scores)
            dict_scores['neutral'] = sum(1 for r in result_scores if r['label'] == 'NEUTRAL') / len(result_scores)
            dict_scores['confidence'] = sum(r['confidence'] for r in result_scores) / len(result_scores)

            dict_scores['is_hallucinated'] = 1.0 if (dict_scores['contradictions'] > 0 or dict_scores['neutral'] > 0.3) else 0
            dict_scores['hallucination_severity'] = (dict_scores['contradictions'] * 1.0) + (dict_scores['neutral'] * 0.5)
            dict_scores['faithfulness_score'] = dict_scores['factual'] - (dict_scores['contradictions'] * 0.5)

            # Calculating BLEU scores
            smoothing = SmoothingFunction()
            var_weights = (0.25, 0.25, 0.25, 0.25)

            dict_scores['bleu_1'] = sentence_bleu([reference_tokens], generated_tokens, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_2'] = sentence_bleu([reference_tokens], generated_tokens, weights=(0.5, 0.5, 0.0, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_3'] = sentence_bleu([reference_tokens], generated_tokens, weights=(0.33, 0.33, 0.33, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_4'] = sentence_bleu([reference_tokens], generated_tokens, weights=(var_weights), smoothing_function=smoothing.method1)
            dict_scores['bleu_avg'] = sum([
                dict_scores['bleu_1'],
                dict_scores['bleu_2'],
                dict_scores['bleu_3'],
                dict_scores['bleu_4']
            ]) / 4.0

            # Calculating smoothing methods
            dict_scores['bleu_smoothing_method0'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method0)
            dict_scores['bleu_smoothing_method1'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method1)
            dict_scores['bleu_smoothing_method2'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method2)
            dict_scores['bleu_smoothing_method3'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method3)
            dict_scores['bleu_smoothing_method4'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method4)
            dict_scores['bleu_smoothing_method5'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method5)
            dict_scores['bleu_smoothing_method6'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method6)
            dict_scores['bleu_smoothing_method7'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method7)

            return dict_scores
        except Exception as e:
            print(f'Error appeared: {str(e)}')
            return dict_scores

    def __get_model_info(self, model_name: str = None) -> Dict[str, Any]:
        """
        Meeting 2 (Extended)
        Private method to extract hardware and model information

        Args:
            model_name (str): Model name

        Returns:
            Dict[str, Any]
        """
        info = {
            'hardware': 'unknown',
            'quantization': 'unknown',
            'param_size': 'unknown',
            'files_size': 'unknown',
            'context_length': 'unknown'
        }
        try:
            if (torch.cuda.is_available()):
                gpu_name = torch.cuda.get_device_name(0)
                info['hardware'] = f'GPU: {gpu_name}'
            else:
                cpu_count = multiprocessing.cpu_count()
                info['hardware'] = f'CPU: {cpu_count} cores'

            if (model_name):
                response = requests.get(f'http://{self.host}:{self.port}/api/tags')
                if (response.status_code == 200):
                    data = response.json()
                    models = data.get('models', [])
                    model = [model for model in models if model_name in model['name']]
                    if model:
                        model = model[0]

                        info['quantization'] = model['details']['quantization_level']
                        info['param_size'] = model['details']['parameter_size']
                        info['files_size'] = f'{round(model['size'] / (1024 ** 3), 3)} GB'
                        info['context_length'] = model['details']['context_length']
        except Exception as e:
            print(f'Exception thrown: {str(e)}')
        finally:
            return info

    def __enter__(self):
        self.start_server()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.stop_server()


if (__name__ == '__main__'):
    i_models = ['tinyllama']
    i_prompt_text = """Summarize this text

Intel Corp. raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning.

The chipmaker priced the offering at $95 per share, according to a company statement. That represents a discount of 6.5% to Friday’s closing price, according to Bloomberg calculations. The share sale drew more than $100 billion in demand, people familiar with the matter said.
"""

    i_generated_text = """Intel Corp. raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning"""
    i_human_label = 'faithful' # faithful or hallucinated
    with OllamaModelTester(host='127.0.0.1', port=11434, models=i_models) as om_tester:
        var_validate_elevator = om_tester.validate_evaluator(
            prompt_text=i_prompt_text,
            generated_text=i_generated_text,
            human_label=i_human_label
        )
        om_tester.print_results([var_validate_elevator])

        om_tester.pull_models(models=None)
        om_tester.compare_models(prompt_text=i_prompt_text, models=None)
        om_tester.print_results()


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Class OllamaModelTester is initialized
Ollama server is started


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1618: UserWarning: Note that pos_label (set to 'hallucinated') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(



human_label: faithful
predicted: faithful
overall_accuracy: 1.0
overall_f1_score: 1.0
hardware: CPU: 2 cores
quantization: unknown
param_size: unknown
files_size: unknown
context_length: unknown
total: 1
factual: 1.0
contradictions: 0.0
neutral: 0.0
confidence: 0.9783622622489929
is_hallucinated: 0
hallucination_severity: 0.0
faithfulness_score: 1.0
bleu_1: 0.14109338070134148
bleu_2: 0.14109338070134148
bleu_3: 0.14109338070134148
bleu_4: 0.14109338070134148
bleu_avg: 0.14109338070134148
bleu_smoothing_method0: 0.14109338070134148
bleu_smoothing_method1: 0.14109338070134148
bleu_smoothing_method2: 0.14109338070134148
bleu_smoothing_method3: 0.14109338070134148
bleu_smoothing_method4: 0.14109338070134148
bleu_smoothing_method5: 0.15756562322202536
bleu_smoothing_method6: 0.14109338070134148
bleu_smoothing_method7: 0.15756562322202536

"tinyllama" model is pulled successfully!
Testing model "tinyllama"
Successfully completed testing model "tinyllama"

model: tinyllama
response: Intel C